<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 2 · Python Data Analysis, Visualization, and Storytelling
### Notebook 02 · Cleaning, Grouping, and Charts

The Titanic file shipped with this book is an aggregated table with
`Class`, `Sex`, `Age`, `Survived`, and `Freq`. That means we clean the
labels, keep the count column, and use `Freq` as the weight when we
summarize survival rates.


### Why this notebook exists
Cleaning and visual analysis belong in the same workflow. This notebook
shows the full path: inspect, clean, summarize, plot, and save the result.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')

data_dir = Path('data')
figures_dir = Path('figures')
artifacts_dir = Path('artifacts')
figures_dir.mkdir(parents=True, exist_ok=True)
artifacts_dir.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(data_dir / 'Titanic.csv')
print(df_raw.head())
print()
print(df_raw.columns.tolist())


In [ ]:
df = df_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]
df = df.rename(columns={'class': 'pclass'})

for col in ['pclass', 'sex', 'age', 'survived']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

df['freq'] = pd.to_numeric(df['freq'], errors='coerce').fillna(0).astype(int)
df['survived_flag'] = df['survived'].str.lower().map({'no': 0, 'yes': 1})
df = df.drop_duplicates()

print(df.head())
print()
print(df.dtypes)


In [ ]:
clean_path = data_dir / 'titanic_clean.csv'
df.to_csv(clean_path, index=False)

def weighted_share(group: pd.DataFrame) -> float:
    weights = group['freq'].to_numpy()
    if weights.sum() == 0:
        return float('nan')
    values = group['survived_flag'].fillna(0).to_numpy()
    return float(np.dot(values, weights) / weights.sum())

survival_by_class = (
    df.groupby('pclass')
    .apply(weighted_share)
    .rename('survival_rate')
    .sort_values(ascending=False)
)

survival_by_sex = (
    df.groupby('sex')
    .apply(weighted_share)
    .rename('survival_rate')
    .sort_values(ascending=False)
)

age_share = (
    df.groupby('age')
    .apply(weighted_share)
    .rename('survival_rate')
    .sort_values(ascending=False)
)

print('survival by class:')
print(survival_by_class)
print()
print('survival by sex:')
print(survival_by_sex)
print()
print('survival by age:')
print(age_share)


In [ ]:
fig, ax = plt.subplots(figsize=(6.3, 3.6))
survival_by_class.plot(kind='bar', ax=ax, color='#0B3C78')
ax.set_ylim(0, 1)
ax.set_ylabel('Survival rate')
ax.set_xlabel('Passenger class')
ax.set_title('Titanic Survival by Passenger Class')
fig.tight_layout()
class_path = figures_dir / 'fig_titanic_survival_by_class.pdf'
fig.savefig(class_path)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6.3, 3.6))
survival_by_sex.plot(kind='bar', ax=ax, color='#0B3C78')
ax.set_ylim(0, 1)
ax.set_ylabel('Survival rate')
ax.set_xlabel('Sex')
ax.set_title('Titanic Survival by Sex')
fig.tight_layout()
sex_path = figures_dir / 'fig_titanic_survival_by_sex.pdf'
fig.savefig(sex_path)
plt.close(fig)

print(class_path.resolve())
print(sex_path.resolve())


In [ ]:
months = np.array(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun'])
signups = np.array([42, 48, 51, 57, 61, 66])

fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(months, signups, marker='o', linewidth=2.0, color='#0B3C78')
ax.set_title('Monthly Signups')
ax.set_xlabel('Month')
ax.set_ylabel('New Signups')
ax.grid(alpha=0.35, linestyle='--', linewidth=0.6)
fig.tight_layout()
trend_path = figures_dir / 'fig_monthly_signups.pdf'
fig.savefig(trend_path)
plt.close(fig)

amounts = np.array([12, 18, 20, 25, 31, 33, 35, 40, 42, 47, 52, 58])
visits = np.array([100, 120, 150, 180, 210, 240])
conversions = np.array([8, 11, 15, 19, 21, 25])

fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.hist(amounts, bins=5, color='#0B3C78', alpha=0.8)
ax.set_title('Order Amounts')
ax.set_xlabel('Amount')
ax.set_ylabel('Count')
fig.tight_layout()
hist_path = figures_dir / 'fig_amount_hist.pdf'
fig.savefig(hist_path)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.scatter(visits, conversions, color='#0B3C78')
ax.set_title('Visits vs Signups')
ax.set_xlabel('Visits')
ax.set_ylabel('Signups')
ax.grid(alpha=0.35, linestyle='--', linewidth=0.6)
fig.tight_layout()
scatter_path = figures_dir / 'fig_visits_vs_signups.pdf'
fig.savefig(scatter_path)
plt.close(fig)

print(trend_path.resolve())
print(hist_path.resolve())
print(scatter_path.resolve())


In [ ]:
summary_path = artifacts_dir / 'titanic_cleaning_summary.txt'
summary_path.write_text(
    'Titanic cleaning summary\n'
    f'Rows: {len(df):,}\n'
    f'Passenger classes: {", ".join(survival_by_class.index.tolist())}\n'
    f'Sex categories: {", ".join(survival_by_sex.index.tolist())}\n'
)

print(summary_path.resolve())


### Where to look next
The next notebook moves from the basic pandas workflow into joins,
reshaping, and method chaining with the small orders tables in this book.
